# 📦 Structured Outputs with Pydantic — Making the LLM Speak Your Data Language

In the [function-calling notebook](./tools_in_llm.ipynb) we taught the model to **ask for help** by calling our functions. There's a second, equally important problem:

> When the model *answers*, **how do we trust the shape of what comes back?**

An LLM speaks *human*. Your code speaks *data* — typed fields, numbers, enums, things you can `if`, `for`, and `+` on. The gap between those two is where production systems quietly break.

This notebook is one long **case study**. We'll build a slice of a real **customer-support automation pipeline** for an e-commerce store, watch it fail in expensive ways, and then fix it with **Pydantic** + the **Responses API**.

We will *not* tour every Pydantic feature. We'll only reach for a feature when a **bug** forces us to.

## The system we're building

Customers email us free text like:

> *"hey my order 10432 arrived smashed, the mug is in pieces. I paid 49.99 for it, want my money back asap"*

Our automation has to turn that into **structured data** and feed three downstream systems:

| Field | Used by | What happens if it's wrong |
|---|---|---|
| `category` | 🧭 the **router** (billing / shipping / technical / ...) | ticket lands in the wrong team's queue — or nowhere |
| `urgency` | ⏱️ the **priority queue** | angry customer waits days; a trivial issue jumps the line |
| `refund_amount` | 💸 the **refund processor** (moves *real money*) | we refund the wrong amount — **literally lose money** |

Three systems. Three different ways to bleed money. Let's watch it happen, then fix it for good.

---
## 0 · Setup

Same boilerplate as the function-calling notebook — load the key, init the client. Nothing new here.

In [15]:
import os, json
from dotenv import load_dotenv
import textwrap

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    print(textwrap.fill(text, width=80))

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found! Check your openai_key.env file.")

# Optional: only needed if you're behind a VPN / corporate proxy (same as before)
import truststore
truststore.inject_into_ssl()

from openai import OpenAI
client = OpenAI(api_key=api_key)

MODEL = "gpt-5-nano"
pretty_print("OpenAI client ready. Model:", MODEL)

OpenAI client ready. Model: gpt-5-nano


In [16]:
# One messy, realistic customer email we'll reuse across the whole notebook.
# Note the signature at the end — we'll need the name + email later.
customer_email = (
    "hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. "
    "I paid 49.99 for it and honestly I'm pretty annoyed, this is the second time. "
    "I just want my money back asap, can you sort this out today?\n\n"
    "Thanks, Sam Rivera (sam.rivera@example.com)"
)
print(customer_email)

hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. I paid 49.99 for it and honestly I'm pretty annoyed, this is the second time. I just want my money back asap, can you sort this out today?

Thanks, Sam Rivera (sam.rivera@example.com)


---
## 1 · The Problem — the model answers in prose, not in fields

Let's ask the model to "extract" the ticket info the way a beginner would: just ask nicely.

In [17]:
response = client.responses.create(
    model=MODEL,
    input=f"Extract the category, urgency and refund amount from this customer message:\n\n{customer_email}",
    reasoning={"effort": "minimal"},
)
print(response.output_text)

- Category: Refund / Refund for damaged item
- Urgency: High (wants resolution today, states “asap” and “sort this out today”)
- Refund amount: 49.99


That reads great to a *human*. But now try to **use** it in code.

- Where's the number? Is it `49.99`, `$49.99`, or "around fifty dollars"?
- Is the category `shipping`, or `"damaged item / shipping"`?
- Run the same cell a few times and the wording **drifts** every time.

Your code can't `if category == ...` against a moving target. Let's make the pain concrete: grab the refund amount and try to do arithmetic on it.

In [18]:
# Our refund processor needs to add a 5% handling fee — i.e. do MATH on the amount.
resp = client.responses.create(
    model=MODEL,
    input=f"In one short line, what refund does this customer want?\n\n{customer_email}",
    reasoning={"effort": "minimal"},
)
raw_amount = resp.output_text
print("Model said:", repr(raw_amount))

# Downstream code expects a number:
try:
    total_with_fee = raw_amount * 1.05   # 💥 string * float
    print("Refund + fee:", total_with_fee)
except Exception as e:
    print("💥 Downstream code blew up:", type(e).__name__, "-", e)

Model said: 'The customer wants a full refund for order 10432 (49.99) due to a damaged item, and wants it processed today.'
💥 Downstream code blew up: TypeError - can't multiply sequence by non-int of type 'float'


---
## 2 · The "just ask for JSON" trap

"Easy," you say, "I'll just tell it to return JSON." Let's try — and be honest about the failure modes.

In [33]:
import concurrent.futures

outputs = []

def fetch_ticket(iteration):
    prompt = f"""Extract the support ticket as JSON with keys: category, urgency, refund_amount.
    Customer message:
    {customer_email}
    """
    resp = client.responses.create(
        model=MODEL,
        input=prompt,
        #reasoning={"effort": "minimal"},
    )
    return iteration, resp.output_text

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(fetch_ticket, i) for i in range(10)]
    for future in concurrent.futures.as_completed(futures):
        iteration, output = future.result()
        outputs.append((iteration, output))
        print(iteration, "Model output:", output)

4 Model output: {
  "category": "Damaged item",
  "urgency": "high",
  "refund_amount": 49.99
}
3 Model output: {
  "category": "damaged_item",
  "urgency": "urgent",
  "refund_amount": 49.99
}
1 Model output: {
  "category": "damaged_item",
  "urgency": "urgent",
  "refund_amount": 49.99
}
0 Model output: {"category":"damaged_item","urgency":"urgent","refund_amount":49.99}
2 Model output: {"category":"damaged_item","urgency":"urgent","refund_amount":49.99}
6 Model output: {"category":"damaged_item","urgency":"high","refund_amount":49.99}
5 Model output: {
  "category": "damaged_item_refund",
  "urgency": "urgent",
  "refund_amount": 49.99
}
8 Model output: {"category":"damaged_item","urgency":"high","refund_amount":49.99}
7 Model output: {
  "category": "damaged_product",
  "urgency": "urgent",
  "refund_amount": 49.99
}
9 Model output: {"category":"damaged_item","urgency":"high","refund_amount":49.99}


Sometimes that's clean JSON. Often it isn't:

- 🪧 it's wrapped in <code>```json ... ```</code> fences, so `json.loads` chokes
- 💬 there's a friendly sentence before it ("Sure! Here's the data:")
- 🔢 `refund_amount` comes back as `"49.99"` (a **string**) or `"$49.99"`
- 🧭 `category` is `"damaged item"` — a label your router has **never heard of**
- 🕳️ a key is silently **missing**

Each is a production incident waiting to happen. Let's parse it the fragile way and watch.

In [34]:
# The naive downstream code: trust the text, json.loads it, act on it.
VALID_CATEGORIES = {"billing", "shipping", "technical", "account", "other"}

def naive_pipeline(model_text):
    data = json.loads(model_text)          # 💥 #1: dies on ``` fences or prose
    category = data["category"]            # 💥 #2: KeyError if missing
    amount = data["refund_amount"]

    # route
    if category not in VALID_CATEGORIES:   # 💥 #3: invented category -> silent misroute
        print(f"  ⚠️  Unknown category {category!r} -> ticket dropped on the floor")
    else:
        print(f"  🧭 routed to: {category}")

    # refund — REAL MONEY
    fee = amount * 1.05                     # 💥 #4: "49.99" * 1.05 explodes; "$49.99" worse
    print(f"  💸 issuing refund (with fee): {fee}")

try:
    naive_pipeline(resp.output_text)
except Exception as e:
    print("💥 pipeline crashed:", type(e).__name__, "-", e)

  ⚠️  Unknown category 'refund' -> ticket dropped on the floor
  💸 issuing refund (with fee): 52.48950000000001


A crash is the *lucky* outcome — at least you find out. The truly expensive bug is the one that **looks fine and runs**.

Imagine the model returns `4999` (it read "49.99" as cents, or just slipped a decimal). It's a valid number. `json.loads` is happy. Your refund processor is happy. You just wired **$5,249** to a customer instead of **$52.49**.

In [21]:
# A perfectly valid-looking JSON... with one quietly catastrophic value
looks_fine = '{"category": "shipping", "urgency": "high", "refund_amount": 4999}'

data = json.loads(looks_fine)          # ✅ no error
refund = data["refund_amount"] * 1.05  # ✅ no error
print(f"💸 Wired ${refund:,.2f} to the customer")   # 😱 should have been ~$52.49
print("Nothing crashed. That's exactly the problem.")

💸 Wired $5,248.95 to the customer
Nothing crashed. That's exactly the problem.


---
## 3 · Enter Pydantic — turn "what we asked for" into a contract

Step back and look at what we kept doing in Sections 1–2: we *described*, in English, the shape we wanted — "give me `category`, `urgency`, `refund_amount`." The model treated that as a **suggestion**, and our `json.loads` blindly trusted whatever came back.

**Pydantic** lets us write that shape down **once**, as a real Python class — not a sentence in a prompt, but an enforceable **contract**. Here's the plan, so you can see where this class is going:

1. **Now:** define a `SupportTicket` class = "this is what a valid ticket looks like."
2. **Now:** run the **model's own output** through it — junk gets rejected *before* it reaches the refund code.
3. **Section 5:** hand that *same class* to the API so the model is **forced** to fill it in.

So `SupportTicket` isn't a random new toy — it's the codified version of the keys we've been begging the model for, and it's the single object that ties this whole notebook together.

In [ ]:
from pydantic import BaseModel

# The contract: a valid support ticket is EXACTLY these three typed fields.
# (These are the very keys we kept asking the model for in our prompts.)
class SupportTicket(BaseModel):
    category: str
    urgency: str
    refund_amount: float

# Picture the model returning clean JSON and us doing json.loads(...) -> this dict.
# Instead of TRUSTING it (like naive_pipeline did), we VALIDATE it against the contract:
model_output = {"category": "shipping", "urgency": "high", "refund_amount": 49.99}

ticket = SupportTicket.model_validate(model_output)   # <- the gate the model's output must pass
print("✅ validated:", ticket)

# refund_amount is now a guaranteed float, so the math that crashed in Section 1 just works:
print("   refund + 5% fee:", round(ticket.refund_amount * 1.05, 2))

✅ validated: category='shipping' urgency='high' refund_amount=49.99
   refund + 5% fee: 52.49


In [23]:
from pydantic import ValidationError

# Section 2 reality #1: the model returned the amount as a STRING. Pydantic coerces it to float:
print(SupportTicket.model_validate(
    {"category": "billing", "urgency": "low", "refund_amount": "49.99"}))

# Section 2 reality #2: the model returned unparseable garbage. Pydantic rejects it — loudly,
# before a single rupee can move:
try:
    SupportTicket.model_validate(
        {"category": "billing", "urgency": "low", "refund_amount": "around fifty bucks"})
except ValidationError as e:
    print("\n🚫 Pydantic blocked the model's junk:\n", e)

category='billing' urgency='low' refund_amount=49.99

🚫 Pydantic blocked the model's junk:
 1 validation error for SupportTicket
refund_amount
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='around fifty bucks', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing


---
## 4 · Tightening the contract — each rule kills a real bug from Section 2

A plain `float` still accepts the model's *worst* outputs: the `4999` that wired **$5,249**, and the invented category `"damaged item"`. Let's make those **impossible to represent**. We're not touring Pydantic's API — we're hardening the contract against the exact incidents we already saw.

### 4.1 · `Literal` + `Field` — the model can't lie in these fields anymore

Our router knows only five categories; refunds have a sane ceiling. Encode both into the **type** itself.

In [24]:
from typing import Literal
from pydantic import Field

class SupportTicket(BaseModel):
    # Only these exact strings exist -> the model can't invent a category
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency:  Literal["low", "medium", "high"]
    # Money is bounded -> the 100x slip can't get through
    refund_amount: float = Field(ge=0, le=500)

# The invented category the model produced in Section 2:
try:
    SupportTicket.model_validate(
        {"category": "damaged item", "urgency": "high", "refund_amount": 49.99})
except ValidationError as e:
    print("🚫 bad category :", e.errors()[0]["msg"])

# The EXACT payload from the silent-bug cell that wired $5,249:
try:
    SupportTicket.model_validate(
        {"category": "shipping", "urgency": "high", "refund_amount": 4999})
except ValidationError as e:
    print("🚫 absurd amount:", e.errors()[0]["msg"])

# A legitimate ticket still passes untouched:
print("✅", SupportTicket.model_validate(
    {"category": "shipping", "urgency": "high", "refund_amount": 49.99}))

🚫 bad category : Input should be 'billing', 'shipping', 'technical', 'account' or 'other'
🚫 absurd amount: Input should be less than or equal to 500
✅ category='shipping' urgency='high' refund_amount=49.99


### 4.2 · Custom rules & nesting — when types aren't enough

Two more requirements real data has:

- An **order id** at our store is always 5 digits. No built-in type says that — a **custom validator** does.
- A ticket carries a nested **customer** (name + email). Pydantic models **compose**, so the model's output stays honest all the way down.

In [25]:
from pydantic import field_validator

class Customer(BaseModel):
    name: str
    email: str

class SupportTicket(BaseModel):
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency:  Literal["low", "medium", "high"]
    refund_amount: float = Field(ge=0, le=500)
    order_id: str = Field(description="The 5-digit order number")
    customer: Customer                       # <-- nested model

    @field_validator("order_id")
    @classmethod
    def must_be_5_digits(cls, v):
        if not (v.isdigit() and len(v) == 5):
            raise ValueError(f"order_id must be exactly 5 digits, got {v!r}")
        return v

# A model payload with a malformed order id is caught:
try:
    SupportTicket.model_validate({
        "category": "shipping", "urgency": "high", "refund_amount": 49.99,
        "order_id": "ABC", "customer": {"name": "Sam", "email": "sam@x.com"},
    })
except ValidationError as e:
    print("🚫", e.errors()[0]["msg"])

# A well-formed model payload validates, and the nested dict becomes a real Customer object:
t = SupportTicket.model_validate({
    "category": "shipping", "urgency": "high", "refund_amount": 49.99,
    "order_id": "10432", "customer": {"name": "Sam", "email": "sam@x.com"},
})
print("✅", t)
print("   customer email:", t.customer.email)   # nested access, fully typed

🚫 Value error, order_id must be exactly 5 digits, got 'ABC'
✅ category='shipping' urgency='high' refund_amount=49.99 order_id='10432' customer=Customer(name='Sam', email='sam@x.com')
   customer email: sam@x.com


### 4.3 · Closing the loop — validate what the model *actually* sends

Everything above used dicts we wrote by hand to *mimic* the model. Let's wire it to the real thing: call the model with plain `responses.create`, then pour its reply straight into our contract with `model_validate_json` (which parses the JSON **and** validates in one step).

This is the honest "manual" way to combine an LLM with Pydantic today.

In [26]:
# Ask the model for JSON (same fragile prompt style as Section 2)...
raw = client.responses.create(
    model=MODEL,
    input=f"Reply with ONLY JSON for these keys: category, urgency, refund_amount, "
          f"order_id, and customer (an object with name and email). "
          f"Customer message:\n\n{customer_email}",
    reasoning={"effort": "minimal"},
).output_text
print("Raw model text:\n", raw, "\n")

# ...then guard it with our contract instead of trusting it:
try:
    ticket = SupportTicket.model_validate_json(raw)   # json.loads + validate, in one call
    print("✅ validated ticket:", ticket)
except Exception as e:
    print("💥 Couldn't validate the raw text:", type(e).__name__, "-", e)
    print("   (the model wrapped it in prose/fences, so our contract never even got to run)")

Raw model text:
 {
  "category": "shipping_damage",
  "urgency": "high",
  "refund_amount": 49.99,
  "order_id": "10432",
  "customer": {
    "name": "Sam Rivera",
    "email": "sam.rivera@example.com"
  }
  } 

💥 Couldn't validate the raw text: ValidationError - 1 validation error for SupportTicket
category
  Input should be 'billing', 'shipping', 'technical', 'account' or 'other' [type=literal_error, input_value='shipping_damage', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error
   (the model wrapped it in prose/fences, so our contract never even got to run)


That worked — **when the model returned clean JSON**. But notice the fragility that's still here:

- if the model adds a sentence or ```` ```json ```` fences, `model_validate_json` never even runs;
- the model can still omit a field or pick a wrong value, and we only find out **after** the round-trip by catching an exception.

We're validating *after the fact*. Wouldn't it be better to **force the model to emit our schema in the first place**? That's exactly what the Responses API does when we hand it our Pydantic class.

---
## 5 · Pydantic + the Responses API — `responses.parse`

Same `SupportTicket` class we just built — but now we give it to the API via **`text_format=`**:

- The SDK turns our class into a **strict JSON schema** and *forces* the model's output to match it — no fences, no prose, no missing keys.
- We get back a **ready-made `SupportTicket` instance** at `response.output_parsed`. The `json.loads` + `model_validate` dance from 4.3 happens for us, and it can't fail the way it did above.

It's the manual bridge from 4.3, made bulletproof. Same messy email from Section 1 — watch.

In [27]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content":
            "Extract the support ticket from the customer message. "
            "Infer urgency from the tone. refund_amount is what the customer paid / wants back "
            "(0 if they aren't asking for a refund)."},
        {"role": "user", "content": customer_email},
    ],
    text_format=SupportTicket,        # <- hand the model OUR class
)

ticket = response.output_parsed        # <- already a validated SupportTicket, not text
print("Type of result:", type(ticket).__name__)
print(ticket)
print()
print("category :", ticket.category)         # a guaranteed-valid Literal
print("urgency  :", ticket.urgency)
print("refund   :", ticket.refund_amount, "->", type(ticket.refund_amount).__name__)
print("order id :", ticket.order_id)
print("customer :", ticket.customer.name, "<" + ticket.customer.email + ">")

Type of result: SupportTicket
category='shipping' urgency='high' refund_amount=49.99 order_id='10432' customer=Customer(name='Sam Rivera', email='sam.rivera@example.com')

category : shipping
urgency  : high
refund   : 49.99 -> float
order id : 10432
customer : Sam Rivera <sam.rivera@example.com>


### 5.1 · The downstream pipeline now *can't* be fed bad data

Remember `naive_pipeline` from Section 2, full of 💥? Here's the same logic — but now every value it receives is **already guaranteed valid**. No defensive checks, no try/except around money. The guarantees moved *up*, into the type.

In [28]:
def process_ticket(ticket: SupportTicket):
    # Every line below is safe because `ticket` couldn't exist if it weren't valid.
    print(f"🧭 routed to    : {ticket.category} team")
    print(f"⏱️  priority     : {ticket.urgency}")
    fee_total = round(ticket.refund_amount * 1.05, 2)   # always a real number in range
    print(f"💸 refund + fee  : ${fee_total:,.2f}")       # sane, every time
    print(f"📦 order         : {ticket.order_id} for {ticket.customer.name}")

process_ticket(ticket)

🧭 routed to    : shipping team
⏱️  priority     : high
💸 refund + fee  : $52.49
📦 order         : 10432 for Sam Rivera


### 5.2 · Even adversarial input yields schema-valid output

What if a customer rambles, jokes, or tries to confuse the bot? The output is *still* a valid `SupportTicket` — the model is constrained to the schema, and Pydantic is the backstop.

In [29]:
weird_email = (
    "lol idk my thing just broke — the login page keeps crashing on the app, "
    "no big deal whenever you get to it. order 88217. - Priya (priya@example.com)"
)

r = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": "Extract the support ticket. Infer urgency from tone. "
                                      "refund_amount is 0 if no refund is requested."},
        {"role": "user", "content": weird_email},
    ],
    text_format=SupportTicket,
)
t2 = r.output_parsed
print(t2, "\n")
process_ticket(t2)   # technical / low / $0.00 — routed correctly, no refund issued

category='technical' urgency='low' refund_amount=0.0 order_id='88217' customer=Customer(name='Priya', email='priya@example.com') 

🧭 routed to    : technical team
⏱️  priority     : low
💸 refund + fee  : $0.00
📦 order         : 88217 for Priya


---
## 6 · The bill — what Pydantic + structured outputs bought us

| Failure mode (Section 2) | Without Pydantic | With Pydantic + `responses.parse` |
|---|---|---|
| prose / ```` ```json ```` fences | `json.loads` crashes randomly | never happens — you get a typed object |
| `refund_amount = "49.99"` (string) | math explodes or corrupts data | coerced to `float`, or rejected |
| `refund_amount = 4999` (100x) | **$5,249 wired by mistake** | rejected by `Field(le=500)` |
| `category = "damaged item"` | silent misroute / dropped ticket | impossible — `Literal` rejects it |
| missing field | `KeyError` deep in the pipeline | rejected up front, clear message |
| `order_id = "ABC"` | bad lookups downstream | rejected by the custom validator |

### The thread we followed
1. **Asked** the model in English → got prose we couldn't use (§1).
2. **Demanded JSON** and hand-parsed it → fragile, and one silent slip cost $5,249 (§2).
3. **Wrote the shape down once** as `SupportTicket`, and validated the model's output through it (§3–4).
4. **Handed that same class to the API** so the model is forced to fill it in (§5).

### The one-sentence takeaway

> **Without a schema, every consumer of the model's output has to defensively re-validate everything — and the one place that forgets is where you lose money.** Pydantic lets you declare *"this is what valid looks like"* exactly once, and holds *both* the model (via `responses.parse`) and your own code to it.

### When to reach for this
- 🧱 Any time the model's output **feeds another system** (DB, API, router, payments).
- 📑 **Extraction** tasks (resumes, invoices, emails → fields).
- 🧪 When you need a **deterministic shape** you can test and monitor.
- 🚫 You can skip it for pure chat / free-form prose meant only for a human to read.